## Bonus-Exercise: Parallelization of Modeling Ocean Circulation with Dask
### Task B.1: Convert the NumPy-based serial implementation to Dask, enabling parallel computation.
In order to parallelize the code with dask we need to make some changes. First we exchange the numpy arrays with dask arrays. In order to calculate the laplacian operator we also need ghost cells, therefore we use map_overlap(). For the boundaries we use the "reflect" mode, since the ocean doesn't have any hard borders and the "constant"-mode would not be suitable.
We also divide the grid in 4 chunks (no matter the grid size) first.

The changes can be seen below:
``` python
import numpy as np
import matplotlib.pyplot as plt
import dask.array as da
import time

def laplacian(field):
    """Computes the discrete Laplacian of a 2D field using finite differences."""
    lap = (
        np.roll(field, shift=1, axis=0) +
        np.roll(field, shift=-1, axis=0) +
        np.roll(field, shift=1, axis=1) +
        np.roll(field, shift=-1, axis=1) -
        4 * field
    )
    return lap

def update_ocean(u, v, temperature, wind, alpha=0.1, beta=0.02):
    """Updates ocean velocity and temperature fields using a simplified flow model."""    
    u_lap = u.map_overlap(laplacian, depth=1, boundary='reflect')
    v_lap = v.map_overlap(laplacian, depth=1, boundary='reflect')
    t_lap = temperature.map_overlap(laplacian, depth=1, boundary='reflect')

    u_new = da.map_blocks(lambda u, ul, w: u + alpha * ul + beta * w, u, u_lap, wind)
    v_new = da.map_blocks(lambda v, vl, w: v + alpha * vl + beta * w, v, v_lap, wind)
    temperature_new = da.map_blocks(lambda t, tl: t + 0.01 * tl, temperature, t_lap)

    return u_new, v_new, temperature_new

# Grid size 
grid_sizes = [200, 2000, 5000]
TIME_STEPS = 100

for grid_size in grid_sizes:
    chunks = (grid_size/2,grid_size/2)
    # Init values
    temperature = da.random.uniform(5, 30, size=(grid_size, grid_size), chunks=chunks)
    u_velocity = da.random.uniform(-1, 1, size=(grid_size, grid_size), chunks=chunks)
    v_velocity = da.random.uniform(-1, 1, size=(grid_size, grid_size), chunks=chunks)
    wind = da.random.uniform(-0.5, 0.5, size=(grid_size, grid_size), chunks=chunks)

    # Run the simulation
    start = time.time()
    for t in range(TIME_STEPS):
        u_velocity, v_velocity, temperature = update_ocean(u_velocity, v_velocity, temperature, wind)
    u_final, v_final, t_final = da.compute(u_velocity, v_velocity, temperature)
    end = time.time()

    print(f"Simulation (grid size: {grid_size}) completed in {end - start} seconds.")
```
Also a timer was added in the original version and the plotting was removed.
Both simulations run with 3 different grid sizes (200, 2000, 5000) in order to compare the execution times.

Let's run both and compare them:

In [1]:
!python ocean_original.py

Simulation (grid size: 200) completed in 0.04039192199707031 seconds.
Simulation (grid size: 2000) completed in 4.406620740890503 seconds.
Simulation (grid size: 5000) completed in 25.57630705833435 seconds.


In [2]:
!python ocean_dask.py

Simulation (grid size: 200) completed in 11.399661779403687 seconds.
Simulation (grid size: 2000) completed in 13.818193912506104 seconds.
Simulation (grid size: 5000) completed in 24.300597190856934 seconds.


As we can see, for the smaller grid the serial version is way faster but with increasing grid sizes the parallel approach with dask is catching up and with a 5000x5000 grid it is even a bit faster. This is due to the overhead dask setting up its parallelization, at some point this overhead is compensated by the speedup of calculating the grid in different portions.
Let's run the dask version again but now lets change it, so the grid is devided in 16 chunks instead of 4.

In [4]:
!python ocean_dask.py

Simulation (grid size: 200) completed in 45.40377902984619 seconds.
Simulation (grid size: 2000) completed in 51.9316349029541 seconds.
Simulation (grid size: 5000) completed in 61.85209107398987 seconds.


The execution times are increasing when deviding the grid in even smaller but more portions. Thats because now even more tasks have to be created and therefore also result in more overhead. This only makes sense if the grid size is very big.
### Task B.2: Monitor Performance with dask.distributed the Dask Dashboard. Use Dask’s Dashboard to analyze execution performance, including CPU usage, memory consumption, and task distribution.